Set up paths and inspect official dataset directories

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import subprocess

REPO_URL = "https://github.com/Sakina152/mm-dg-healthcare.git"
REPO_DIR = "/content/mm-dg-healthcare"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
!git log -1 --oneline


Mounted at /content/drive
Working directory: /content/mm-dg-healthcare
7c0583f (HEAD -> main, origin/main, origin/HEAD) Update 08


In [6]:

from pathlib import Path
import requests
from html.parser import HTMLParser
from urllib.parse import urljoin
import pandas as pd

REPO = Path("/content/mm-dg-healthcare")
PTB_DIR = REPO / "data/raw/ptbxl"
OUT = REPO / "experiments/results/lead_observability_pilot"
OUT.mkdir(parents=True, exist_ok=True)

assert PTB_DIR.joinpath("ptbxl_database.csv").exists()
assert PTB_DIR.joinpath("scp_statements.csv").exists()

BASE = "https://physionet.org/files/challenge-2021/1.0.3/training/"

DATASETS = {
    "chapman_shaoxing": BASE + "chapman_shaoxing/g1/",
    "cpsc_2018": BASE + "cpsc_2018/g1/",
    "georgia": BASE + "georgia/g1/",
}

class DirectoryParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.hrefs = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "a":
            for key, value in attrs:
                if key.lower() == "href" and value:
                    self.hrefs.append(value)

def inspect_directory(url):
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    parser = DirectoryParser()
    parser.feed(response.text)

    files = sorted({
        href.rsplit("/", 1)[-1]
        for href in parser.hrefs
        if href.lower().endswith((".hea", ".mat", ".dat"))
    })

    return files

for dataset, url in DATASETS.items():
    try:
        files = inspect_directory(url)
        print(f"\n{dataset}: {url}")
        print("Files found:", len(files))
        print("Examples:", files[:8])
    except Exception as exc:
        print(f"\n{dataset}: directory inspection failed")
        print(repr(exc))



chapman_shaoxing: https://physionet.org/files/challenge-2021/1.0.3/training/chapman_shaoxing/g1/
Files found: 1998
Examples: ['JS00001.hea', 'JS00001.mat', 'JS00002.hea', 'JS00002.mat', 'JS00004.hea', 'JS00004.mat', 'JS00005.hea', 'JS00005.mat']

cpsc_2018: https://physionet.org/files/challenge-2021/1.0.3/training/cpsc_2018/g1/
Files found: 1998
Examples: ['A0001.hea', 'A0001.mat', 'A0002.hea', 'A0002.mat', 'A0003.hea', 'A0003.mat', 'A0004.hea', 'A0004.mat']

georgia: https://physionet.org/files/challenge-2021/1.0.3/training/georgia/g1/
Files found: 1998
Examples: ['E00001.hea', 'E00001.mat', 'E00002.hea', 'E00002.mat', 'E00003.hea', 'E00003.mat', 'E00004.hea', 'E00004.mat']


In [3]:

from pathlib import Path
import tarfile
import pandas as pd

REPO = Path("/content/mm-dg-healthcare")
ARCHIVE = Path("/content/drive/MyDrive/mm-dg-healthcare-data/ptbxl.tar")
RAW = REPO / "data" / "raw"

assert REPO.exists(), f"Repository missing: {REPO}"
assert ARCHIVE.exists(), f"Drive archive missing: {ARCHIVE}"

RAW.mkdir(parents=True, exist_ok=True)

with tarfile.open(ARCHIVE, "r:*") as tar:
    # Prevent archive paths from escaping the destination directory.
    for member in tar.getmembers():
        target = (RAW / member.name).resolve()
        if not target.is_relative_to(RAW.resolve()):
            raise ValueError(f"Unsafe archive path: {member.name}")

    tar.extractall(RAW, filter="data")

PTB_DIR = RAW / "ptbxl"
db_path = PTB_DIR / "ptbxl_database.csv"
scp_path = PTB_DIR / "scp_statements.csv"

assert db_path.exists(), f"Metadata not found: {db_path}"
assert scp_path.exists(), f"SCP mapping not found: {scp_path}"

df = pd.read_csv(db_path)
scp = pd.read_csv(scp_path, index_col=0)

print("PTB-XL restored successfully.")
print("PTB-XL directory:", PTB_DIR)
print("Records:", len(df))
print("SCP statements:", len(scp))
print("Database shape:", df.shape)


PTB-XL restored successfully.
PTB-XL directory: /content/mm-dg-healthcare/data/raw/ptbxl
Records: 21799
SCP statements: 71
Database shape: (21799, 28)


Inspect the official diagnosis labels and signal headers

In [7]:

from pathlib import Path
import requests

HEADER_DIR = OUT / "sample_headers"
HEADER_DIR.mkdir(parents=True, exist_ok=True)

for dataset, url in DATASETS.items():
    try:
        files = inspect_directory(url)
        headers = [name for name in files if name.lower().endswith(".hea")]

        if not headers:
            print(f"{dataset}: no headers listed")
            continue

        name = headers[0]
        response = requests.get(urljoin(url, name), timeout=60)
        response.raise_for_status()

        path = HEADER_DIR / f"{dataset}_{name}"
        path.write_bytes(response.content)

        print(f"\n--- {dataset} / {name} ---")
        print(response.text[:2500])

    except Exception as exc:
        print(f"{dataset}: failed: {exc}")



--- chapman_shaoxing / JS00001.hea ---
JS00001 12 500 5000
JS00001.mat 16x1+24 1000.0(0)/mV 16 0 -254 21756 0 I
JS00001.mat 16x1+24 1000.0(0)/mV 16 0 264 -599 0 II
JS00001.mat 16x1+24 1000.0(0)/mV 16 0 517 -22376 0 III
JS00001.mat 16x1+24 1000.0(0)/mV 16 0 -5 28232 0 aVR
JS00001.mat 16x1+24 1000.0(0)/mV 16 0 -386 16619 0 aVL
JS00001.mat 16x1+24 1000.0(0)/mV 16 0 390 15121 0 aVF
JS00001.mat 16x1+24 1000.0(0)/mV 16 0 -98 1568 0 V1
JS00001.mat 16x1+24 1000.0(0)/mV 16 0 -312 -32761 0 V2
JS00001.mat 16x1+24 1000.0(0)/mV 16 0 -98 32715 0 V3
JS00001.mat 16x1+24 1000.0(0)/mV 16 0 810 15193 0 V4
JS00001.mat 16x1+24 1000.0(0)/mV 16 0 810 14081 0 V5
JS00001.mat 16x1+24 1000.0(0)/mV 16 0 527 32579 0 V6
# Age: 85
# Sex: Male
# Dx: 164889003,59118001,164934002
# Rx: Unknown
# Hx: Unknown
# Sx: Unknown


--- cpsc_2018 / A0001.hea ---
A0001 12 500 7500
A0001.mat 16x1+24 1000.0(0)/mV 16 0 28 -1716 0 I
A0001.mat 16x1+24 1000.0(0)/mV 16 0 7 2029 0 II
A0001.mat 16x1+24 1000.0(0)/mV 16 0 -21 3745 0 III
A0

Save the audit report to Google Drive

In [8]:

from pathlib import Path
import shutil
from google.colab import drive

drive.mount("/content/drive")

BACKUP = Path(
    "/content/drive/MyDrive/mm-dg-healthcare-data/"
    "results_backup/lead_observability_pilot"
)
BACKUP.mkdir(parents=True, exist_ok=True)

# Back up sample headers
if HEADER_DIR.exists():
    shutil.copytree(
        HEADER_DIR,
        BACKUP / "sample_headers",
        dirs_exist_ok=True
    )

print("Pilot audit files backed up to:", BACKUP)
for p in sorted(BACKUP.rglob("*")):
    if p.is_file():
        print(p.relative_to(BACKUP), f"({p.stat().st_size:,} bytes)")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pilot audit files backed up to: /content/drive/MyDrive/mm-dg-healthcare-data/results_backup/lead_observability_pilot
sample_headers/chapman_shaoxing_JS00001.hea (760 bytes)
sample_headers/cpsc_2018_A0001.hea (700 bytes)
sample_headers/georgia_E00001.hea (713 bytes)


Audit the dataset metadata systematically

In [9]:

from pathlib import Path
from urllib.parse import urljoin
from html.parser import HTMLParser
import requests
import pandas as pd
import re

class LinkParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "a":
            for key, value in attrs:
                if key.lower() == "href" and value:
                    self.links.append(value)

def list_links(url):
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    parser = LinkParser()
    parser.feed(response.text)
    return sorted(set(parser.links))

def parse_header_text(text, dataset, filename):
    lines = text.splitlines()
    first = lines[0].split() if lines else []

    if len(first) < 4:
        return None

    try:
        n_leads = int(first[1])
        fs = float(first[2])
        n_samples = int(first[3])
    except (ValueError, IndexError):
        return None

    lead_names = []
    for line in lines[1:]:
        parts = line.split()
        if len(parts) >= 2:
            lead_names.append(parts[-1])

    def comment_value(key):
        pattern = rf"^\s*#{key}\s*:\s*(.*)$"
        for line in lines:
            match = re.match(pattern, line, flags=re.IGNORECASE)
            if match:
                return match.group(1).strip()
        return None

    duration = n_samples / fs if fs > 0 else None

    return {
        "dataset": dataset,
        "header": filename,
        "n_leads": n_leads,
        "sampling_hz": fs,
        "n_samples": n_samples,
        "duration_seconds": duration,
        "lead_names": ";".join(lead_names),
        "diagnosis_codes": comment_value("Dx"),
        "age": comment_value("Age"),
        "sex": comment_value("Sex"),
    }

AUDIT_DIR = OUT / "metadata_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

rows = []

for dataset, url in DATASETS.items():
    print(f"Auditing {dataset} ...")

    links = list_links(url)
    headers = sorted(
        link.rsplit("/", 1)[-1]
        for link in links
        if link.lower().endswith(".hea")
    )

    print("Header files listed:", len(headers))

    # Sample up to 50 headers for a quick metadata audit.
    # This is a pilot, not a full-dataset inventory.
    for filename in headers[:50]:
        try:
            response = requests.get(urljoin(url, filename), timeout=60)
            response.raise_for_status()

            row = parse_header_text(
                response.text, dataset, filename
            )
            if row is not None:
                rows.append(row)

        except Exception as exc:
            print("Header failed:", filename, repr(exc))

audit_df = pd.DataFrame(rows)

csv_path = AUDIT_DIR / "external_metadata_sample_audit.csv"
audit_df.to_csv(csv_path, index=False)

print("\nSaved:", csv_path)
display(audit_df.groupby("dataset").agg(
    sampled_records=("header", "count"),
    lead_counts=("n_leads", lambda s: sorted(s.unique().tolist())),
    sampling_rates=("sampling_hz", lambda s: sorted(s.unique().tolist())),
    durations=("duration_seconds", lambda s: sorted(s.unique().tolist())),
))


Auditing chapman_shaoxing ...
Header files listed: 999
Auditing cpsc_2018 ...
Header files listed: 999
Auditing georgia ...
Header files listed: 999

Saved: /content/mm-dg-healthcare/experiments/results/lead_observability_pilot/metadata_audit/external_metadata_sample_audit.csv


,sampled_records,lead_counts,sampling_rates,durations
dataset,,,,
chapman_shaoxing,50,[12],[500.0],[10.0]
cpsc_2018,50,[12],[500.0],"[10.0, 10.554, 10.568, 10.994, 11.0, 11.614, 1..."
georgia,50,[12],[500.0],[10.0]


Inspect the available dataset groups and diagnosis mapping files

In [10]:

from urllib.parse import urljoin
import pandas as pd

print("Dataset group directories and mapping files")

for dataset, url in DATASETS.items():
    parent_url = url.rsplit("g1/", 1)[0]
    try:
        links = list_links(parent_url)
        names = sorted(link.rstrip("/").split("/")[-1] for link in links)

        groups = [
            name for name in names
            if name.startswith("g") and name[1:].isdigit()
        ]

        mapping_files = [
            name for name in names
            if any(key in name.lower() for key in [
                "mapping", "diagnos", "snomed", "label", "reference"
            ])
        ]

        print(f"\n{dataset}")
        print("Group folders:", groups)
        print("Potential mapping/reference files:", mapping_files[:30])

    except Exception as exc:
        print(dataset, "directory inspection failed:", repr(exc))


Dataset group directories and mapping files

chapman_shaoxing
Group folders: ['g1', 'g10', 'g11', 'g2', 'g3', 'g4', 'g5', 'g6', 'g7', 'g8', 'g9']
Potential mapping/reference files: []

cpsc_2018
Group folders: ['g1', 'g2', 'g3', 'g4', 'g5', 'g6', 'g7']
Potential mapping/reference files: []

georgia
Group folders: ['g1', 'g10', 'g11', 'g2', 'g3', 'g4', 'g5', 'g6', 'g7', 'g8', 'g9']
Potential mapping/reference files: []


In [11]:

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://physionet.org/files/challenge-2021/1.0.3/training/"

DATASETS = [
    "chapman_shaoxing",
    "cpsc_2018",
    "georgia",
]

session = requests.Session()
rows = []

for dataset in DATASETS:
    parent_url = BASE + dataset + "/"
    response = session.get(parent_url, timeout=60)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    groups = sorted(
        {
            link.get("href", "").strip("/")
            for link in soup.find_all("a")
            if link.get("href", "").strip("/").startswith("g")
            and link.get("href", "").strip("/")[1:].isdigit()
        },
        key=lambda name: int(name[1:])
    )

    print(f"\n{dataset}: {len(groups)} groups")

    total_headers = 0

    for group in groups:
        url = f"{parent_url}{group}/"
        r = session.get(url, timeout=60)
        r.raise_for_status()

        group_soup = BeautifulSoup(r.text, "html.parser")
        count = sum(
            1 for a in group_soup.find_all("a")
            if a.get("href", "").lower().endswith(".hea")
        )

        total_headers += count
        rows.append({
            "dataset": dataset,
            "group": group,
            "header_count": count,
        })

        print(f"  {group}: {count} headers")

    print("Total headers:", total_headers)

group_audit = pd.DataFrame(rows)
display(group_audit)

out = (
    "/content/mm-dg-healthcare/experiments/results/"
    "lead_observability_pilot/group_coverage_audit.csv"
)
group_audit.to_csv(out, index=False)
print("Saved:", out)



chapman_shaoxing: 11 groups
  g1: 999 headers
  g2: 1000 headers
  g3: 1000 headers
  g4: 1000 headers
  g5: 1000 headers
  g6: 1000 headers
  g7: 1000 headers
  g8: 1000 headers
  g9: 1000 headers
  g10: 1000 headers
  g11: 248 headers
Total headers: 10247

cpsc_2018: 7 groups
  g1: 999 headers
  g2: 1000 headers
  g3: 1000 headers
  g4: 1000 headers
  g5: 1000 headers
  g6: 1000 headers
  g7: 878 headers
Total headers: 6877

georgia: 11 groups
  g1: 999 headers
  g2: 1000 headers
  g3: 1000 headers
  g4: 1000 headers
  g5: 1000 headers
  g6: 1000 headers
  g7: 1000 headers
  g8: 1000 headers
  g9: 1000 headers
  g10: 1000 headers
  g11: 345 headers
Total headers: 10344


,dataset,group,header_count
0,chapman_shaoxing,g1,999
1,chapman_shaoxing,g2,1000
2,chapman_shaoxing,g3,1000
3,chapman_shaoxing,g4,1000
4,chapman_shaoxing,g5,1000
5,chapman_shaoxing,g6,1000
6,chapman_shaoxing,g7,1000
7,chapman_shaoxing,g8,1000
8,chapman_shaoxing,g9,1000
9,chapman_shaoxing,g10,1000


Saved: /content/mm-dg-healthcare/experiments/results/lead_observability_pilot/group_coverage_audit.csv


In [12]:

from pathlib import Path
import requests
import pandas as pd

mapping_dir = Path(
    "/content/mm-dg-healthcare/experiments/results/"
    "lead_observability_pilot/label_mapping"
)
mapping_dir.mkdir(parents=True, exist_ok=True)

base = (
    "https://raw.githubusercontent.com/"
    "physionetchallenges/evaluation-2021/master/"
)

for filename in [
    "dx_mapping_scored.csv",
    "dx_mapping_unscored.csv",
]:
    response = requests.get(base + filename, timeout=60)
    response.raise_for_status()

    path = mapping_dir / filename
    path.write_bytes(response.content)

    df = pd.read_csv(path)
    print(f"\n{filename}: {df.shape}")
    print("Columns:", df.columns.tolist())
    display(df.head())

print("\nMapping files saved to:", mapping_dir)



dx_mapping_scored.csv: (30, 13)
Columns: ['Dx', 'SNOMEDCTCode', 'Abbreviation', 'CPSC', 'CPSC_Extra', 'StPetersburg', 'PTB', 'PTB_XL', 'Georgia', 'Chapman_Shaoxing', 'Ningbo', 'Total', 'Notes']


,Dx,SNOMEDCTCode,Abbreviation,CPSC,CPSC_Extra,StPetersburg,PTB,PTB_XL,Georgia,Chapman_Shaoxing,Ningbo,Total,Notes
0,atrial fibrillation,164889003,AF,1221,153,2,15,1514,570,1780,0,5255,NaN
1,atrial flutter,164890007,AFL,0,54,0,1,73,186,445,7615,8374,NaN
2,bundle branch block,6374002,BBB,0,0,1,20,0,116,0,385,522,NaN
3,bradycardia,426627000,Brady,0,271,11,0,0,6,0,7,295,NaN
4,complete left bundle branch block,733534002,CLBBB,0,0,0,0,0,0,0,213,213,We score 733534002 and 164909002 as the same d...



dx_mapping_unscored.csv: (103, 12)
Columns: ['Dx', 'SNOMEDCTCode', 'Abbreviation', 'CPSC', 'CPSC_Extra', 'StPetersburg', 'PTB', 'PTB_XL', 'Georgia', 'Chapman_Shaoxing', 'Ningbo', 'Total']


,Dx,SNOMEDCTCode,Abbreviation,CPSC,CPSC_Extra,StPetersburg,PTB,PTB_XL,Georgia,Chapman_Shaoxing,Ningbo,Total
0,accelerated atrial escape rhythm,233892002,AAR,0,0,0,0,0,0,0,16,16
1,abnormal QRS,164951009,abQRS,0,0,0,0,3389,0,0,0,3389
2,atrial escape beat,251187003,AED,0,0,0,0,0,0,0,17,17
3,accelerated idioventricular rhythm,61277005,AIVR,0,0,0,0,0,0,0,14,14
4,accelerated junctional rhythm,426664006,AJR,0,0,0,0,0,19,0,12,31



Mapping files saved to: /content/mm-dg-healthcare/experiments/results/lead_observability_pilot/label_mapping


Identify the best shared diagnosis labels

In [13]:

from pathlib import Path
import pandas as pd

RESULTS = Path(
    "/content/mm-dg-healthcare/experiments/results/"
    "lead_observability_pilot"
)
MAPPING_DIR = RESULTS / "label_mapping"

scored = pd.read_csv(MAPPING_DIR / "dx_mapping_scored.csv")

columns = [
    "Dx",
    "SNOMEDCTCode",
    "Abbreviation",
    "PTB_XL",
    "Georgia",
    "Chapman_Shaoxing",
    "CPSC",
]

counts = scored[columns].copy()

count_cols = ["PTB_XL", "Georgia", "Chapman_Shaoxing", "CPSC"]
for col in count_cols:
    counts[col] = pd.to_numeric(counts[col], errors="coerce").fillna(0)

# These are mapping-table counts, not yet verified counts from our loaded files.
counts["minimum_shared_count"] = counts[count_cols].min(axis=1)
counts["total_across_four"] = counts[count_cols].sum(axis=1)

# Require at least 100 mapped examples in every source for this initial shortlist.
shortlist = counts[
    (counts[count_cols] >= 100).all(axis=1)
].sort_values("minimum_shared_count", ascending=False)

print("Scored classes:", len(counts))
print("Classes with >=100 mapped records in all four sources:", len(shortlist))
display(shortlist)

out = RESULTS / "shared_scored_label_shortlist.csv"
shortlist.to_csv(out, index=False)
print("Saved:", out)


Scored classes: 30
Classes with >=100 mapped records in all four sources: 5


,Dx,SNOMEDCTCode,Abbreviation,PTB_XL,Georgia,Chapman_Shaoxing,CPSC,minimum_shared_count,total_across_four
13,sinus rhythm,426783006,NSR,18092,1752,1826,918,918,22588
0,atrial fibrillation,164889003,AF,1514,570,1780,1221,570,5085
14,premature atrial contraction,284470004,PAC,398,639,258,616,258,1911
6,1st degree av block,270492004,IAVB,797,769,247,722,247,2535
10,left bundle branch block,164909002,LBBB,536,231,205,236,205,1208


Saved: /content/mm-dg-healthcare/experiments/results/lead_observability_pilot/shared_scored_label_shortlist.csv


In [14]:

from pathlib import Path
from google.colab import drive
import shutil

drive.mount("/content/drive")

source_dir = Path(
    "/content/mm-dg-healthcare/experiments/results/lead_observability_pilot"
)

backup_dir = Path(
    "/content/drive/MyDrive/mm-dg-healthcare-data/"
    "results_backup/lead_observability_pilot"
)
backup_dir.mkdir(parents=True, exist_ok=True)

files_to_backup = [
    source_dir / "shared_scored_label_shortlist.csv",
    source_dir / "group_coverage_audit.csv",
]

for src in files_to_backup:
    if not src.exists():
        print("MISSING:", src.name)
        continue

    dst = backup_dir / src.name
    shutil.copy2(src, dst)

    if src.stat().st_size == dst.stat().st_size:
        print(f"Verified: {dst.name} ({dst.stat().st_size:,} bytes)")
    else:
        print(f"SIZE MISMATCH: {dst.name}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verified: shared_scored_label_shortlist.csv (416 bytes)
Verified: group_coverage_audit.csv (602 bytes)
